## Unit 10. 偏微分方程式 (PDE) 課堂作業
- 課程編號: CHEXXXX
- 學年度: 114下
- 上課時間: 每週四 09:00-12:00
-
- 指導教授: ＯＯＯ 教授
- 學生姓名: ＯＯＯ
- 學號: m12345678
- email帳號: fcu.m12345678@gmail.com

---
### 0. 環境設定

In [ ]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'Unit10_Homework'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

try:
    shortcut_path = '/content/ChemE-3502'
    os.remove(shortcut_path)
except (FileNotFoundError, OSError):
    pass

if IN_COLAB:
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3502')
    os.symlink(source_path, shortcut_path)
    shortcut_path = Path(shortcut_path)
    if source_path.exists():
        NOTEBOOK_DIR = shortcut_path / 'Unit10'
        OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
        FIG_DIR      = OUTPUT_DIR / 'figs'
    else:
        print("⚠️ 找不到雲端 ChemE-3502 路徑，請確認資料夾名稱是否正確")
else:
    NOTEBOOK_DIR = Path.cwd()
    OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    FIG_DIR      = OUTPUT_DIR / 'figs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

---
### 1. 載入套件

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 繪圖樣式設定
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'lines.linewidth': 2,
    'axes.unicode_minus': False,
})

print("✓ 套件載入完成")
print(f"  numpy  版本: {np.__version__}")
import scipy
print(f"  scipy  版本: {scipy.__version__}")
import matplotlib
print(f"  matplotlib 版本: {matplotlib.__version__}")
try:
    import pde
    print(f"  py-pde 版本: {pde.__version__}")
except ImportError:
    print("  ⚠️ py-pde 未安裝，請執行: pip install py-pde")

---
## I. 練習題一：葡萄糖偵測器薄膜內的濃度分布
**（改編自 ch5 習題5-4-2）**

### 問題描述

考慮一葡萄糖偵測器中，以高分子膠原薄膜為載體，其上固定含有葡糖氧化酵素之薄層。此系統將葡萄糖轉化為雙氧水，藉由電極電流推論葡萄糖濃度。假設酵素表面反應為一次反應，薄膜上葡萄糖濃度之無因次化 PDE 方程式為：

$$
\frac{\partial C}{\partial \theta} = \frac{\partial^2 C}{\partial x^2}
$$

其中無因次時間 $\theta = Dt/\delta^2$，無因次距離 $x = X/\delta$，$D$ 為擴散係數，$\delta$ 為薄膜厚度。

**起始條件：**

$$
C(x, 0) = 0, \quad 0 \leq x \leq 1
$$

**邊界條件：**

$$
C(0, \theta) = 1 \quad (\text{Dirichlet, } \theta > 0)
$$

$$
\left.\frac{\partial C}{\partial x}\right|_{x=1} = -\mu C \quad (\text{Robin, } \theta > 0)
$$

其中 Damkoehler 數 $\mu = K\delta/D$，$K$ 為表面反應速率常數。

**已知參數：**
- 擴散係數 $D = 2 \times 10^{-6}\ \text{cm}^2/\text{s}$
- 薄膜厚度 $\delta = 0.03\ \text{cm}$
- 表面反應速率常數 $K = 0.24\ \text{cm/h}$

> **提示：** $\mu = K\delta/D$，請先計算 $\mu$ 的數值。Robin 邊界條件可採用 Method of Lines (MoL) 實作或以 `py-pde` 自訂邊界條件求解。

### 1.1 計算 Damkoehler 數 $\mu$

計算此系統之 Damkoehler 數 $\mu$，並說明其物理意義（反應速率與擴散速率之比值）。

In [ ]:
# ========================================
# 1.1 計算 Damkoehler 數 μ
# ========================================
# 已知參數
D = 2e-6          # 擴散係數 [cm²/s]
delta = 0.03      # 薄膜厚度 [cm]
K = 0.24 / 3600   # 表面反應速率常數 [cm/s]  (原為 cm/h，轉換為 cm/s)

# 計算 Damkoehler 數
mu = K * delta / D

print(f"D     = {D:.2e} cm²/s")
print(f"delta = {delta} cm")
print(f"K     = {K:.4e} cm/s  (= 0.24 cm/h)")
print(f"\nDamkoehler 數 μ = K·δ/D = {mu:.4f}")
print("\n物理意義說明:")
# 請在下方填入你對 μ 物理意義的說明
# ===== 請填入你的分析 =====
print("μ = (表面反應速率) / (薄膜中之擴散速率)")

### 1.2 使用 Method of Lines (MoL) 求解薄膜濃度分布

使用 `scipy.integrate.solve_ivp()` 配合空間有限差分離散化求解此 PDE。
- 空間節點數建議：$N = 50$，無因次時間範圍：$\theta \in [0, 5]$
- 正確處理 Robin 邊界條件：$\partial C/\partial x|_{x=1} = -\mu C$
- 輸出不同 $\theta$ 時刻之濃度分布 $C(x, \theta)$

In [ ]:
from scipy.integrate import solve_ivp

# ========================================
# 1.2 Method of Lines 求解葡萄糖偵測器薄膜PDE
# ========================================
# 無因次參數
# mu 已由 1.1 計算得到

# 空間離散化


def pde_rhs(theta, C_inner):
    """
    計算 dC/dθ 之右側 (Method of Lines)
    C_inner: 內部節點濃度 C[1..N]
    邊界條件:
        左端 (x=0): C[0] = 1         (Dirichlet)
        右端 (x=1): dC/dx = -μ*C     (Robin)
          -> 以後向差分近似: (C[N+1] - C[N]) / dx = -mu * C[N]
          -> C_right = C[N] - mu*dx*C[N] = C[N]*(1 - mu*dx)
    """


    return

# 起始條件: C(x,0) = 0 for 0 < x ≤ 1


# 時間範圍


# 求解


# ===== 請在此處繼續實作結果分析 =====

### 1.3 結果視覺化與分析

繪製以下圖表並進行分析：
1. 不同無因次時刻 $\theta$ 之薄膜內濃度分布曲線（$C$ vs $x$）
2. 酵素表面（$x = 1$）處濃度 $C(1, \theta)$ 隨時間的變化曲線
3. 說明穩態時濃度分布的物理意義：$\mu$ 的大小如何影響穩態濃度梯度？

In [ ]:
# ========================================
# 1.3 結果視覺化
# ========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 圖1: 不同時刻之濃度分布曲線 ---


# --- 圖2: 酵素表面濃度 C(x=1) 隨時間的變化 ---


# ===== 請在下方填入你對結果的分析說明 =====
print("\n[分析] 請描述：")
print("1. 濃度分布隨時間的演變趨勢")
print("2. μ 值大小對穩態濃度梯度的影響")

### 1.4 參數敏感度分析：比較不同 $\mu$ 值的影響

分別取 $\mu = 0.1, 1.0, 5.0, 10.0$，在穩態（$\theta = 5$）時，比較薄膜內的濃度分布差異。
- 請在同一張圖中繪製四條穩態濃度曲線
- 說明 $\mu$ 值增大（反應速率相對擴散速率加快）時，濃度分布如何變化

In [ ]:
# ========================================
# 1.4 不同 μ 值之穩態濃度分布比較
# ========================================
mu_list = [0.1, 1.0, 5.0, 10.0]
theta_ss = 5.0   # 近似穩態時刻


# ===== 請在下方填入你的分析說明 =====
print("\n[你的分析]")
print("請說明 μ 值增大時，穩態濃度分布如何變化，以及其物理意義...")

---
## II. 練習題二：牛頓流體受制於移動平板之速度分布
**（改編自 ch5 習題5-4-3，Stokes First Problem）**

### 問題描述

大量的水（水溫 $25^\circ\text{C}$）靜置於一無限大水平平板上。在某一瞬間，平板以速度 $v = 0.2\ \text{m/s}$ 突然開始移動，求水在垂直方向（$y$ 方向）之速度分布隨時間的演變。

根據連續方程式與動量平衡方程式，一維流場之統御 PDE 為：

$$
\frac{\partial v_x}{\partial t} = \nu \frac{\partial^2 v_x}{\partial y^2}, \quad \nu = \frac{\mu}{\rho}
$$

**起始條件：**

$$
v_x(y, 0) = 0, \quad \forall y \geq 0
$$

**邊界條件：**

$$
v_x(0, t) = v = 0.2\ \text{m/s} \quad (\text{Dirichlet, 移動平板})
$$

$$
v_x(y_{\max}, t) \approx 0 \quad (\text{遠場近似 } y \to \infty)
$$

**已知流體物性（水, 25°C）：**
- 動力黏度 $\mu = 8.931 \times 10^{-4}\ \mathrm{kg/(m \cdot s)}$
- 密度 $\rho = 994.6\ \text{kg/m}^3$
- 平板速度 $v = 0.2\ \text{m/s}$

**解析解（誤差函數解）：**

$$
\frac{v_x}{v} = 1 - \text{erf}\!\left(\frac{y}{2\sqrt{\nu t}}\right) = \text{erfc}\!\left(\frac{y}{2\sqrt{\nu t}}\right)
$$

> **提示：** $y_{\max}$ 應選取足夠大（如 $y_{\max} = 0.05\ \text{m}$），以確保邊界速度趨近於零。

### 2.1 計算運動黏度 $\nu$ 與特徵擴散時間

計算水在 $25^\circ\text{C}$ 下的運動黏度 $\nu = \mu/\rho$，並估算速度邊界層厚度 $\delta(t) \approx 4\sqrt{\nu t}$ 在 $t = 30\ \text{s}$ 時的值。

In [ ]:
# ========================================
# 2.1 計算運動黏度與邊界層厚度
# ========================================


# 運動黏度


# 估算不同時刻之邊界層厚度


# 建議的計算域高度 (確保速度趨近0)


### 2.2 使用 py-pde 求解 Stokes 第一問題

使用 `py-pde` 的 `DiffusionPDE`（擴散係數 = $\nu$）求解此動量擴散 PDE。
- 計算域：$y \in [0,\ y_{\max}]$，網格點數 $N = 100$
- 時間範圍：$t \in [0,\ 60]\ \text{s}$，記錄 $t = 5, 10, 20, 40, 60\ \text{s}$ 時的速度分布
- 邊界條件：$v_x(0,t) = 0.2\ \text{m/s}$（Dirichlet），$v_x(y_{\max},t) = 0$（Dirichlet）
- 與解析解 $v_x/v = \text{erfc}\!\left(y/(2\sqrt{\nu t})\right)$ 比對

In [ ]:
import pde
from scipy.special import erfc

# ========================================
# 2.2 py-pde 求解 Stokes 第一問題
# ========================================


# ========================================
# 視覺化: 數值解 vs 解析解
# ========================================


### 2.3 誤差分析與結果討論

1. 計算各時刻數值解與解析解之最大絕對誤差 $\max|v_{x,\text{num}} - v_{x,\text{analytic}}|$
2. 觀察速度邊界層隨時間之發展，說明為何速度擾動越來越深入流體
3. 解釋此 PDE 為何與熱傳導方程式在數學上完全等價（動量擴散 vs 熱量擴散）

In [ ]:
# ========================================
# 2.3 誤差分析
# ========================================


# ===== 請在下方填入你的分析討論 =====
print("\n[你的分析討論]")
print("請回答：")
print("1. 數值解與解析解的誤差量級？是否在可接受範圍內？")
print("2. 速度邊界層隨時間的發展原因？")
print("3. 此問題與熱傳導的相似性說明？")

---
## III. 練習題三：二維平板穩態熱傳
**（改編自 ch5 習題5-4-8，2D Steady-State Heat Conduction）**

### 問題描述

考慮一薄矩形鋼板，長 $L_x = 20\ \text{cm}$，寬 $L_y = 10\ \text{cm}$。邊界條件如下：
- $x = L_x = 20\ \text{cm}$（右側邊）：$T = 100^\circ\text{C}$
- 其餘三邊（$x=0$，$y=0$，$y=L_y$）：$T = 0^\circ\text{C}$

試求穩態時鋼板內部的二維溫度分布。

**統御方程式（Laplace 方程式，橢圓型 PDE）：**

$$
\frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2} = 0
$$

**邊界條件：**

$$
T(0,\, y) = 0, \quad T(L_x,\, y) = 100^\circ\text{C}, \quad T(x,\, 0) = 0, \quad T(x,\, L_y) = 0
$$

> **提示：**
> - 方法一：使用 `py-pde` 的 `CartesianGrid` (2D) 時間推進至穩態（令求解時間足夠長）
> - 方法二：使用 `scipy.linalg.solve()` 對有限差分離散化之線性系統直接求解（矩陣法）
> - 解析解（Fourier 級數）：$T(x,y) = \sum_{n=1,3,5,...}^{\infty} \frac{4 \cdot 100}{n\pi} \frac{\sinh(n\pi x/L_y)}{\sinh(n\pi L_x/L_y)} \sin(n\pi y/L_y)$

### 3.1 方法一：py-pde 時間推進法求解穩態 Laplace 方程式

使用 `py-pde` 的 `LaplacePDE`（or 等效 `DiffusionPDE`）配合 `CartesianGrid` (2D) 時間推進求解。
- 網格：$N_x = 40$，$N_y = 20$（對應 $\Delta x = 0.5\ \text{cm}$）
- 推進至足夠長時間 $t_f$ 以達穩態（建議觀察殘差收斂）
- 輸出穩態溫度場並繪製等溫線圖

In [ ]:
import pde

# ========================================
# 3.1 py-pde 時間推進法求穩態 2D 溫度場
# ========================================
Lx = 20.0   # 板長 [cm]
Ly = 10.0   # 板寬 [cm]
Nx = 40     # x 方向網格數
Ny = 20     # y 方向網格數
T_right = 100.0   # 右側邊溫度 [°C]
T_other = 0.0     # 其餘三邊溫度 [°C]

# 建立2D網格: x ∈ [0, Lx], y ∈ [0, Ly]


# 初始溫度場 (均勻 0°C)


# 邊界條件設定
# x 方向: 左端 x=0 → T=0 (Dirichlet), 右端 x=Lx → T=100 (Dirichlet)
# y 方向: 下端 y=0 → T=0 (Dirichlet), 上端 y=Ly → T=0 (Dirichlet)


# 使用 LaplacePDE (= DiffusionPDE with diffusivity=1 in steady-state sense)
# 此處以 DiffusionPDE 時間推進至穩態


# 時間推進至穩態 (大的 t 值)


# ========================================
# 繪製穩態溫度場
# ========================================


# 等溫線圖 (contour)


# 3D 曲面圖


### 3.2 方法二：scipy 有限差分矩陣法（直接求解線性系統）

將二維 Laplace 方程式在矩形網格上進行有限差分離散化，建立線性方程組 $\mathbf{A}\vec{T} = \vec{b}$，以 `scipy.linalg.solve()` 直接求解。

- 將內部網格點 $(i, j)$ 之二維索引轉換為一維向量索引（`idx = i * Ny + j`）
- 建立係數矩陣 $\mathbf{A}$（$N_{\text{inner}} \times N_{\text{inner}}$ 稀疏矩陣）及右側向量 $\vec{b}$
- 求解後重新排列為二維溫度場，與方法一結果比較

In [ ]:
from scipy.linalg import solve as sp_solve

# ========================================
# 3.2 scipy 有限差分矩陣法求解 Laplace 方程式
# ========================================
# 使用相同的網格尺寸


# 建立係數矩陣 A 和右側向量 b


# 求解線性系統


# 重新排列為二維陣列 (含邊界)


# 繪製兩種方法的比較


### 3.3 與解析解比較及結果分析

1. 利用 Fourier 級數解析解計算準確解，並與兩種數值方法比較最大誤差
2. 計算板中心點 $(10\ \text{cm},\ 5\ \text{cm})$ 的溫度，並與解析解比較
3. 說明兩種數值方法的優缺點（計算效率、程式複雜度、適用性）

In [ ]:
# ========================================
# 3.3 Fourier 級數解析解比較
# ========================================
def T_analytic_2d(x, y, Lx=20.0, Ly=10.0, T_wall=100.0, n_terms=50):
    """
    Laplace 方程式解析解 (Fourier 級數)
    邊界條件: 右側 T=T_wall, 其餘三邊 T=0
    T(x,y) = Σ (4*T_wall / nπ) * [sinh(nπx/Ly) / sinh(nπLx/Ly)] * sin(nπy/Ly)
             n = 1, 3, 5, ...
    """


# 在 py-pde 網格上計算解析解


# 計算板中心點溫度


# 從 py-pde 結果插值得到中心點溫度


# 繪製解析解


# ===== 請填入你的分析 =====
print("\n[你的分析討論]")
print("請比較：")
print("1. 兩種數值方法與解析解的誤差")
print("2. 兩種方法的計算效率與程式複雜度優缺點")

---
## IV. 額外加分作業（自由繳交）
- 學生可自由選擇是否繳交加分作業（下週上課前完成 email 電子檔）
- 分數會加在最後總成績上，每份加 0.1 ~ 1.0 分
- 加分作業不一定要全部都完成，完成其中一項即可繳交
- 請務必自行完成！你可以問 AI、問同學，但一定要自行**吸收、執行、整理**結果
- 不要直接複製 AI 的答案寄給老師
- 如果系統比對發現作業內容雷同，原創性低，作業分數為 0 分

---
### 加分題：球形催化劑顆粒中的耦合 PDE 求解
**（改編自 ch5 習題5-4-5）**

在某一球形顆粒催化劑中，質量與熱量平衡之耦合動態方程式（球座標，無因次化）為：

$$
\frac{\partial y}{\partial t} = \frac{1}{r^2}\frac{\partial}{\partial r}\!\left(r^2 \frac{\partial y}{\partial r}\right) - \phi^2 R(y,\theta)
$$

$$
\alpha \frac{\partial \theta}{\partial t} = \frac{1}{r^2}\frac{\partial}{\partial r}\!\left(r^2 \frac{\partial \theta}{\partial r}\right) + \beta\phi^2 R(y,\theta)
$$

其中無因次反應速率 $R = \exp\!\left[\gamma\!\left(1 - 1/\theta\right)\right] y$，$y$ 為無因次濃度，$\theta$ 為無因次溫度。

**邊界條件：**

$$
\left.\frac{\partial y}{\partial r}\right|_{r=0} = \left.\frac{\partial \theta}{\partial r}\right|_{r=0} = 0 \quad (\text{球心對稱})
$$

$$
y|_{r=1} = \theta|_{r=1} = 1 \quad (\text{表面 Dirichlet})
$$

**起始條件：** $y|_{t=0} = 0$，$\theta|_{t=0} = 1$，$0 \leq r \leq 1$

**已知參數：** $\phi = 1$，$\beta = 0.05$，$\gamma = 15$，$\alpha = 1$

**要求：**
1. 使用 `py-pde` 的 `SphericalGrid` 建立耦合 PDE 系統求解
2. 繪製不同時刻之濃度 $y(r,t)$ 與溫度 $\theta(r,t)$ 徑向分布曲線
3. 繪製球心（$r=0$）與表面（$r=1$）處的時間響應曲線
4. 討論：$\beta$ 值（放熱程度）對溫度-濃度分布的影響

In [ ]:


# ===== 請在下方填入你的分析 =====
print("\n[你的分析討論]")
print("請說明：")
print("1. 濃度與溫度分布的動態特徵")
print("2. β 值增大（放熱增強）對濃度/溫度分布的影響")


---
## 💭 思考題

請在課堂作業中尋找你認為最有趣的一個問題，並回答以下思考題（文字說明即可，不需要程式碼）：

1. **PDE 分類的工程意義**：橢圓型、拋物線型與雙曲線型 PDE 分別對應什麼樣的物理現象？在化工問題中各舉一個實際案例。

2. **邊界條件的影響**：在葡萄糖偵測器問題（練習題一）中，Robin 邊界條件與單純 Dirichlet 或 Neumann 條件相比，有何物理意義上的差異？如果酵素表面反應速率 $K \to \infty$，邊界條件會退化為哪一類型？

3. **數值方法選擇**：對於同一個 PDE 問題，py-pde 時間推進法與 scipy 矩陣直接法各有哪些優缺點？在哪些情況下你會選擇哪種方法？

4. **物理類比**：Stokes 第一問題（練習題二）的統御方程式與熱傳導方程式在數學上完全相同，這種「動量-熱量類比」在化工設計中有什麼實際應用價值？

5. **自由發揮**：在課堂演練的範例（Unit10_Example_01 至 06）中，哪一個案例讓你印象最深刻？請說明其工程背景及求解過程中最困難的部分。

**[請在此填入你的思考題回答]**

1. 

2. 

3. 

4. 

5. 

---
# 想對老師說的話